# **Fine-Tune Pre-Trained Classification Model With QLORA**

### **Import Libraries**

In [4]:
import torch
import torch.nn as nn   
import json
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification 

In [3]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
device.type

'cuda'

In [ ]:
def save_data_to_file (data, file_path):
    
    with open(file_path, "w") as file:
        json.dump(data, file,indent=4)
    print(f"file created successfully at {file_path}")

In [6]:
def load_data_from_file(file_path):
    
    with open(file_path, "r") as file:
        load_file = json.load(file)
    
    return load_file

### **Load Dataset**

In [7]:
dataset = load_dataset("imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
train_labels = dataset['train']['label']
num_classes = len(set(train_labels))
num_classes

2

In [9]:
imdb_labels = {0: 'negative review', 1: 'positive review'}

In [10]:
# load tokenizer
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Vish\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


##### Tokenizer Map Function

In [12]:
def tokenized_text (data):
    return tokenizer(data['text'], return_tensors='pt', 
                     padding='max_length', max_length=512, truncation=True)

In [13]:
dataset = dataset.map(tokenized_text, batched=True)

Map: 100%|██████████| 50000/50000 [00:45<00:00, 1100.84 examples/s]


In [14]:
dataset['train'][0]

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be